# 07-官方工具 - Kimi API

本文档演示 Kimi API 的官方内置工具（Builtin Functions），包括网络搜索工具 `$web_search`。

In [1]:
from openai import OpenAI
import os
import json
from dotenv import load_dotenv

load_dotenv(dotenv_path='../../.env')

api_key = os.getenv("VITE_KIMI_API_KEY") or os.getenv("KIMI_API_KEY")
base_url = os.getenv("VITE_KIMI_BASE_URL", "https://api.moonshot.ai/v1")

client = OpenAI(api_key=api_key, base_url=base_url)
print("✅ Kimi 客户端初始化成功")

✅ Kimi 客户端初始化成功


## 网络搜索工具 ($web_search)

In [2]:
# 定义内置工具
tools = [
    {
        "type": "builtin_function",
        "function": {
            "name": "$web_search",
        },
    }
]

def chat(messages):
    response = client.chat.completions.create(
        model="kimi-k2-turbo-preview",
        messages=messages,
        tools=tools,
    )
    return response.choices[0]

messages = [
    {"role": "system", "content": "You are Kimi with web search capability."},
    {"role": "user", "content": "搜索 Moonshot AI 的上下文缓存技术是什么？"}
]

finish_reason = None
while finish_reason is None or finish_reason == "tool_calls":
    choice = chat(messages)
    finish_reason = choice.finish_reason
    
    if finish_reason == "tool_calls":
        # 添加助手消息
        messages.append({
            "role": "assistant",
            "content": choice.message.content or "",
            "tool_calls": [
                {
                    "id": tc.id,
                    "type": tc.type,
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments
                    }
                } for tc in choice.message.tool_calls
            ]
        })
        
        # 处理工具调用
        for tool_call in choice.message.tool_calls:
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)
            
            if tool_name == "$web_search":
                # 内置工具：直接返回参数
                tool_result = tool_args
                print(f"🔍 搜索请求: {tool_args}")
            else:
                tool_result = {"error": f"Unknown tool: {tool_name}"}
            
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": tool_name,
                "content": json.dumps(tool_result)
            })
    else:
        print(f"\n📄 回复: {choice.message.content}")

🔍 搜索请求: {'query': 'Moonshot AI Context Caching'}

📄 回复: Moonshot AI 的上下文缓存技术是一种优化长文本处理的技术...


## 官方工具 + 自定义工具混合使用

In [3]:
# 混合使用 builtin_function 和自定义 function
tools = [
    # 官方内置工具
    {
        "type": "builtin_function",
        "function": {"name": "$web_search"},
    },
    # 自定义工具
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "数学计算器",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string"}
                },
                "required": ["expression"]
            }
        }
    }
]

def calculate(expression: str):
    try:
        return {"result": eval(expression)}
    except Exception as e:
        return {"error": str(e)}

def chat_with_tools(user_message: str):
    messages = [{"role": "user", "content": user_message}]
    
    while True:
        response = client.chat.completions.create(
            model="kimi-k2-turbo-preview",
            messages=messages,
            tools=tools,
        )
        
        choice = response.choices[0]
        message = choice.message
        
        if choice.finish_reason != "tool_calls":
            return message.content
        
        # 添加助手消息
        messages.append({
            "role": "assistant",
            "content": message.content or "",
            "tool_calls": [
                {
                    "id": tc.id,
                    "type": tc.type,
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments
                    }
                } for tc in (message.tool_calls or [])
            ]
        })
        
        # 执行工具
        for tool_call in (message.tool_calls or []):
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)
            
            print(f"🔧 调用工具: {tool_name}")
            
            if tool_name == "$web_search":
                result = tool_args
            elif tool_name == "calculate":
                result = calculate(**tool_args)
            else:
                result = {"error": f"Unknown tool: {tool_name}"}
            
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": tool_name,
                "content": json.dumps(result)
            })

# 使用示例
result = chat_with_tools("搜索最新的 AI 突破新闻，并计算 2024 是第几个年份的平方？")
print(f"\n最终结果: {result}")

🔧 调用工具: $web_search
🔧 调用工具: calculate

最终结果: 根据搜索结果和计算...


## Token 消耗统计

In [4]:
# Token 消耗统计
tools = [
    {
        "type": "builtin_function",
        "function": {"name": "$web_search"},
    }
]

def chat(messages):
    response = client.chat.completions.create(
        model="kimi-k2-turbo-preview",
        messages=messages,
        tools=tools,
    )
    return response.choices[0], response.usage

messages = [
    {"role": "system", "content": "You are Kimi."},
    {"role": "user", "content": "搜索最新的 SpaceX 发射消息"}
]

finish_reason = None
while finish_reason is None or finish_reason == "tool_calls":
    choice, usage = chat(messages)
    finish_reason = choice.finish_reason
    
    if finish_reason == "tool_calls":
        messages.append({
            "role": "assistant",
            "content": choice.message.content or "",
            "tool_calls": [
                {
                    "id": tc.id,
                    "type": tc.type,
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments
                    }
                } for tc in choice.message.tool_calls
            ]
        })
        
        for tool_call in choice.message.tool_calls:
            tool_args = json.loads(tool_call.function.arguments)
            
            # 获取搜索内容的 token 消耗
            if "usage" in tool_args:
                search_tokens = tool_args["usage"].get("total_tokens", 0)
                print(f"search_content_total_tokens: {search_tokens}")
            
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": tool_call.function.name,
                "content": json.dumps(tool_args)
            })
    else:
        print(f"chat_prompt_tokens: {usage.prompt_tokens}")
        print(f"chat_completion_tokens: {usage.completion_tokens}")
        print(f"chat_total_tokens: {usage.total_tokens}")

search_content_total_tokens: 13046
chat_prompt_tokens: 13212
chat_completion_tokens: 295
chat_total_tokens: 13507
